In [1]:
# ============================================================
# DATASET: LendingClub
# Purpose: Test SMOTE-ENN as an alternative to plain SMOTE,
#          per the approved plan's Risk 3 mitigation. Comparing
#          against the existing no-SMOTE and plain-SMOTE results.
# ============================================================

import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier
import re

SCALED_DIR = Path("../Data/Processed")
RANDOM_SEED = 42

def clean_column_names(df):
    df.columns = [re.sub(r"[\[\]<>]", "_", str(col)) for col in df.columns]
    return df

X_lending_train = pd.read_csv(SCALED_DIR / "lending_X_train.csv")
clean_column_names(X_lending_train)
X_lending_test = pd.read_csv(SCALED_DIR / "lending_X_test.csv")
clean_column_names(X_lending_test)
y_lending_train = pd.read_csv(SCALED_DIR / "lending_y_train.csv").squeeze()
y_lending_test = pd.read_csv(SCALED_DIR / "lending_y_test.csv").squeeze()

print(X_lending_train.shape, X_lending_test.shape)

(8000, 69) (2000, 69)


In [2]:
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "config": name,
        "AUC-ROC": roc_auc_score(y_test, y_proba),
        "F1 (minority)": f1_score(y_test, y_pred),
        "G-mean": geometric_mean_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "positive_predictions": int(y_pred.sum())  # tracking this specifically,
        # since RF+SMOTE's defining failure was predicting near-zero positives
    }

In [3]:
# RF with SMOTE-ENN (combines oversampling with cleaning of overlapping/noisy samples -
# potentially avoids the synthetic-sample confusion that caused plain SMOTE to fail RF here)
rf_smoteenn = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200))
])
result_rf_smoteenn = evaluate_model(rf_smoteenn, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "RF (SMOTE-ENN)")

# XGBoost with SMOTE-ENN, for comparison
xgb_smoteenn = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])
result_xgb_smoteenn = evaluate_model(xgb_smoteenn, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "XGBoost (SMOTE-ENN)")

smoteenn_results = pd.DataFrame([result_rf_smoteenn, result_xgb_smoteenn])
print(smoteenn_results)

                config   AUC-ROC  F1 (minority)    G-mean       MCC  \
0       RF (SMOTE-ENN)  0.812627        0.00000  0.000000 -0.003028   
1  XGBoost (SMOTE-ENN)  0.881930        0.54902  0.623451  0.598484   

   positive_predictions  
0                     1  
1                    15  


In [4]:
# ============================================================
# DATASET: German Credit
# Purpose: SMOTE-ENN test, for completeness against the other
#          two datasets and the existing no-SMOTE/plain-SMOTE results
# ============================================================

X_german_train = pd.read_csv(SCALED_DIR / "german_X_train.csv")
clean_column_names(X_german_train)
X_german_test = pd.read_csv(SCALED_DIR / "german_X_test.csv")
clean_column_names(X_german_test)
y_german_train = pd.read_csv(SCALED_DIR / "german_y_train.csv").squeeze()
y_german_test = pd.read_csv(SCALED_DIR / "german_y_test.csv").squeeze()

rf_smoteenn_german = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200))
])
result_rf_smoteenn_german = evaluate_model(rf_smoteenn_german, X_german_train, y_german_train, X_german_test, y_german_test, "RF (SMOTE-ENN)")

xgb_smoteenn_german = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])
result_xgb_smoteenn_german = evaluate_model(xgb_smoteenn_german, X_german_train, y_german_train, X_german_test, y_german_test, "XGBoost (SMOTE-ENN)")

print(pd.DataFrame([result_rf_smoteenn_german, result_xgb_smoteenn_german]))

                config   AUC-ROC  F1 (minority)    G-mean       MCC  \
0       RF (SMOTE-ENN)  0.789524       0.635135  0.744264  0.452801   
1  XGBoost (SMOTE-ENN)  0.770952       0.589041  0.704661  0.379068   

   positive_predictions  
0                    88  
1                    86  


In [5]:
# ============================================================
# DATASET: Home Credit
# Purpose: SMOTE-ENN test, for completeness
# ============================================================

X_home_train = pd.read_csv(SCALED_DIR / "home_X_train.csv")
clean_column_names(X_home_train)
X_home_test = pd.read_csv(SCALED_DIR / "home_X_test.csv")
clean_column_names(X_home_test)
y_home_train = pd.read_csv(SCALED_DIR / "home_y_train.csv").squeeze()
y_home_test = pd.read_csv(SCALED_DIR / "home_y_test.csv").squeeze()

rf_smoteenn_home = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200))
])
result_rf_smoteenn_home = evaluate_model(rf_smoteenn_home, X_home_train, y_home_train, X_home_test, y_home_test, "RF (SMOTE-ENN)")

xgb_smoteenn_home = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss"))
])
result_xgb_smoteenn_home = evaluate_model(xgb_smoteenn_home, X_home_train, y_home_train, X_home_test, y_home_test, "XGBoost (SMOTE-ENN)")

print(pd.DataFrame([result_rf_smoteenn_home, result_xgb_smoteenn_home]))

                config   AUC-ROC  F1 (minority)    G-mean       MCC  \
0       RF (SMOTE-ENN)  0.722291       0.181841  0.372534  0.135032   
1  XGBoost (SMOTE-ENN)  0.746494       0.234824  0.421157  0.196459   

   positive_predictions  
0                  2921  
1                  2794  
